# Course 5: Winning Online AI Competitions
## Session 6: Time series and small data

**Goal:** forecast 28 days ahead with rolling-origin validation, and compare models on a small tabular dataset.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on synthetic or built-in data. Cells labelled NOT RUN need a Kaggle account, a competition's data, a GPU or an extra library.

### 1. Daily sales for 20 products
Synthetic data with a weekly rhythm, a yearly rhythm, a trend and product-specific levels. The task, as in the M5 competition: forecast each of the next 28 days.

In [ ]:
import numpy as np, pandas as pd, warnings, lightgbm as lgb
warnings.filterwarnings("ignore")
rng = np.random.default_rng(0); T, P, H = 730, 20, 28
day = np.arange(T); level = rng.uniform(20, 100, P); week = rng.uniform(0.1, 0.4, P); trend = rng.normal(0.0003, 0.0003, P)
promo = rng.random((T, P)) < 0.05
mu = level * (1 + week * np.sin(2 * np.pi * day[:, None] / 7 + rng.uniform(0, 6, P))) * (1 + 0.2 * np.sin(2 * np.pi * day[:, None] / 365)) * (1 + trend * day[:, None]) * (1 + 0.5 * promo)
sales = rng.poisson(mu).astype(float)
print("shape (days, products):", sales.shape)

### 2. Rolling-origin validation
Pick several cut-off dates. For each, train on everything before it and forecast the next 28 days. Averaging over origins tells you how the method does at different times, which a single split cannot.

In [ ]:
origins = [T - 28 * k for k in (4, 3, 2, 1)]                          # four forecast origins, 28 days apart
def evaluate(forecast_fn):
    errs = []
    for o in origins:
        pred = forecast_fn(sales[:o], promo[:o + H], o)                # promotions are planned, so future values are known
        errs.append(np.abs(pred - sales[o:o + H]).mean() / np.abs(sales[7:o] - sales[:o - 7]).mean())    # scaled by the in-sample seasonal-naive error
    return np.mean(errs), np.std(errs)

### 3. Baselines first
- Seasonal naive: each day equals the same weekday four weeks back, repeated.
- Weekday mean: the average of the last 8 weeks for that weekday.

In [ ]:
def seasonal_naive(hist, promo_all, o): return np.vstack([hist[-28:]])
def weekday_mean(hist, promo_all, o):
    last = hist[-56:].reshape(8, 7, -1).mean(0); return np.vstack([last[(i) % 7] for i in range(H)])
results = {"seasonal naive": evaluate(seasonal_naive), "weekday mean (8 weeks)": evaluate(weekday_mean)}
for k, v in results.items(): print(f"{k:26s} scaled error {v[0]:.3f} ± {v[1]:.3f}")

### 4. A tree model with lag features
One row per (product, day). The catch: to forecast 28 days ahead in one shot, a feature may only use data that is at least 28 days old at prediction time. Lags of 28, 35, 42 and rolling means that start 28 days back are safe. A lag of 1 is not.

In [ ]:
def make_rows(hist, promo_all, days, safe=True):
    rows = []
    for d in days:
        for p in range(P):
            L = 28 if safe else 1                                      # the shortest lag used
            past = hist[:d - L + 1, p] if d - L + 1 <= len(hist) else None
            f = {"product": p, "dow": d % 7, "doy": d % 365, "promo": int(promo_all[d, p]), "t": d,
                 "lag_a": hist[d - L, p], "lag_b": hist[d - L - 7, p], "lag_c": hist[d - L - 14, p],
                 "roll28": hist[d - L - 27:d - L + 1, p].mean(), "same_dow4": np.mean([hist[d - (28 if safe else 7) - 7 * k, p] for k in range(4)])}
            rows.append(f)
    return pd.DataFrame(rows)

def gbdt(hist, promo_all, o):
    train_days = range(120, o)
    Xtr = make_rows(hist, promo_all, train_days); ytr = np.array([hist[d, p] for d in train_days for p in range(P)])
    m = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=31, objective="poisson", verbose=-1, random_state=0).fit(Xtr, ytr)
    # At prediction time, every feature for day o + h must come from days before o. Lag 28 of day o + h is day o + h - 28 < o: safe.
    padded = np.vstack([hist, np.full((H, P), np.nan)])
    Xte = make_rows(padded, promo_all, range(o, o + H))
    assert not Xte.isna().any().any(), "a feature used data from the forecast period"
    return m.predict(Xte).reshape(H, P)
results["LightGBM, lags of 28+ days"] = evaluate(gbdt)
for k, v in results.items(): print(f"{k:26s} scaled error {v[0]:.3f} ± {v[1]:.3f}")
assert results["LightGBM, lags of 28+ days"][0] < results["seasonal naive"][0]

The assertion inside `gbdt` is the important line. If a feature needs a day that has not happened yet, the padded history returns NaN and the check fails. Build this guard into every forecasting pipeline.

### 5. The classic mistake: a lag of 1
With yesterday's sales as a feature, validation on rows with known history works. At real forecast time, yesterday is unknown for 27 of the 28 days, so most of the feature values do not exist.

In [ ]:
o = origins[-1]; hist = sales[:o]
Xtr = make_rows(hist, promo, range(120, o - 28), safe=False); ytr = np.array([hist[d, p] for d in range(120, o - 28) for p in range(P)])
m1 = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, objective="poisson", verbose=-1, random_state=0).fit(Xtr, ytr)
Xva = make_rows(hist, promo, range(o - 28, o), safe=False)                                   # validation rows: yesterday is known here
looks = np.abs(m1.predict(Xva).reshape(28, P) - hist[o - 28:o]).mean() / np.abs(sales[7:o] - sales[:o - 7]).mean()
padded = np.vstack([hist, np.full((H, P), np.nan)]); Xte = make_rows(padded, promo, range(o, o + H), safe=False)
print(f"lag-1 model on validation rows with known history: scaled error {looks:.3f}  (the safe model above: {results['LightGBM, lags of 28+ days'][0]:.3f})")
print(f"at real forecast time, share of feature values that do not exist yet: {Xte[['lag_a', 'lag_b', 'lag_c', 'roll28']].isna().mean().mean():.0%}")

The fixes are to use only safe lags (as above), to train one model per horizon, or to forecast recursively and accept that errors compound.

### 6. Small data: when trees are not the answer
On a few hundred rows, a regularized linear model or a tabular foundation model can beat boosted trees. Compare on the breast cancer dataset at several training sizes, with repeated splits.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score
X, y = load_breast_cancer(return_X_y=True)
models = {"logistic regression": lambda: make_pipeline(StandardScaler(), LogisticRegression(C=0.5, max_iter=2000)),
          "LightGBM": lambda: lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=8, min_child_samples=5, verbose=-1, random_state=0)}
print("training rows   " + "   ".join(f"{k:>20s}" for k in models))
for n_train in [30, 100, 300]:
    row = []
    for name, make in models.items():
        aucs = [roc_auc_score(y[te], make().fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]) for tr, te in StratifiedShuffleSplit(20, train_size=n_train, test_size=200, random_state=0).split(X, y)]
        row.append(f"{np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"{n_train:13d}   " + "   ".join(f"{r:>20s}" for r in row))

### 7. TabPFN (NOT RUN here)

In [ ]:
# NOT RUN: needs `pip install tabpfn`, a model download, and ideally a GPU. Check the package's current limits on rows and features and its licence.
from tabpfn import TabPFNClassifier
# models["TabPFN"] = lambda: TabPFNClassifier()
# Rerun the comparison above. TabPFN is a transformer pretrained on synthetic tables; it predicts in one forward pass with no tuning.
# It was built for small tables, so compare it at 30, 100 and 300 rows, and include the time each model takes.

### TODO
1. Add a fifth origin and a hierarchy: also score the forecast of total sales across products. Does the best model change?
2. Train one model per horizon week (days 1-7 use lags from 7, days 8-14 from 14, and so on). Does it beat the single model?
3. Run TabPFN in section 6 and add a column to the table.